In [0]:
#build two tables, silver.maintenance_events (one row per entry) and silver.maintenance_summary (one row per log)
from pyspark.sql import functions as F

C = "capstone_supply_chain"
logs = spark.table(f"{C}.bronze.raw_maintenance_logs")


def extract(col, pattern):
    """Text captured by the (group), or NULL if the pattern is not found."""
    v = F.regexp_extract(col, pattern, 1)
    return F.when(v == "", F.lit(None).cast("string")).otherwise(F.trim(v))


def extract_int(col, pattern):
    v = F.regexp_extract(col, pattern, 1)
    return F.when(v != "", v.cast("int"))


def extract_num(col, pattern):
    """Number captured by the (group), with thousands separators removed."""
    v = F.regexp_replace(F.regexp_extract(col, pattern, 1), ",", "")
    return F.when(v != "", v.cast("double"))


def minutes(text_col):
    """'2h 15min' -> 135, '45min' -> 45, '4h 00min' -> 240."""
    return (F.coalesce(extract_int(text_col, r"(\d+)h"), F.lit(0)) * 60
            + F.coalesce(extract_int(text_col, r"(\d+)\s*min"), F.lit(0)))


def sum_dollars(col):
    """Add up every $ amount mentioned in a text."""
    amounts = F.regexp_extract_all(col, F.lit(r"\$([0-9,]+)"), F.lit(1))
    return F.aggregate(
        amounts, F.lit(0.0),
        lambda acc, x: acc + F.when(x != "", F.regexp_replace(x, ",", "").cast("double")).otherwise(F.lit(0.0)))


# 1. Split each log into its entries and its summary
raw = F.col("raw_content")
parts = (logs
    .withColumn("summary_text", F.regexp_extract(raw, r"(?s)([A-Z]+ \d{4} SUMMARY:.*)$", 1))
    .withColumn("body_text", F.regexp_replace(raw, r"(?s)[A-Z]+ \d{4} SUMMARY:.*$", ""))
    .withColumn("plant_code", extract(raw, r"PLANT (PLT-[A-Z0-9]+)"))
    .withColumn("line_name", extract(raw, r"(ASSEMBLY LINE \d+)"))
)

# 2. One row per ENTRY
t = F.col("entry_text")
events = (parts
    .withColumn("entry_text", F.explode(F.split("body_text", r"(?=ENTRY \d{3} \|)")))
    .filter(F.col("entry_text").startswith("ENTRY"))
    .withColumn("entry_no", extract(t, r"ENTRY (\d{3})"))
    .withColumn("entry_date", F.expr("try_cast(regexp_extract(entry_text, 'ENTRY \\\\d{3} \\\\| (\\\\d{4}-\\\\d{2}-\\\\d{2})', 1) AS DATE)"))
    .withColumn("shift", extract(t, r"\| Shift ([A-Z])"))
    .withColumn("technician", extract(t, r"Tech: ([^\n]+)"))
    .withColumn("equipment_id", extract(t, r"Equipment: (?:[A-Za-z]+ )?([A-Z0-9]+(?:-[A-Z0-9]+)+)"))
    .withColumn("equipment_line", extract(t, r"Equipment: ([^\n]+)"))
    .withColumn("equipment_model", extract(t, r"Equipment: \S+ \(([^)]+)\)"))
    .withColumn("maintenance_type", extract(t, r"Type: ([^|\n]+?)\s*\|"))
    .withColumn("duration_text", extract(t, r"Duration: ([^\n]+)"))
    .withColumn("duration_min", minutes(F.col("duration_text")))
    .withColumn("issue", extract(t, r"Issue: ([^\n]+)"))
    .withColumn("root_cause", extract(t, r"Root Cause: ([^\n]+)"))
    .withColumn("action", extract(t, r"Action: ([^\n]+)"))
    .withColumn("impact", extract(t, r"Impact: ([^\n]+)"))
    .withColumn("is_unplanned", F.col("maintenance_type").startswith("Unplanned"))
    .withColumn("fault_code", extract(F.col("issue"), r"\b(SRVO-\d+|F\d{4})\b"))
    .withColumn("units_delayed", extract_int(F.col("impact"), r"(\d+) units delayed"))
    .withColumn("scrap_parts", extract_int(F.col("impact"), r"(\d+) parts scrapped"))
    .withColumn("scrap_cost_usd", extract_num(F.col("impact"), r"scrapped \(\$([0-9,]+)\)"))
    .withColumn("parts_cost_listed_usd", sum_dollars(F.col("action")))
    .select("plant_code", "line_name", "entry_no", "entry_date", "shift", "technician",
            "equipment_id", "equipment_line", "equipment_model", "maintenance_type", "is_unplanned",
            "duration_text", "duration_min", "fault_code", "issue", "root_cause",
            "action", "impact", "units_delayed", "scrap_parts", "scrap_cost_usd",
            "parts_cost_listed_usd", "filename",
            F.current_timestamp().alias("silver_processed_at"))
)
(events.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{C}.silver.maintenance_events"))

# 3. One row per SUMMARY
s = F.col("summary_text")
summary = (parts.filter(F.col("summary_text") != "")
    .withColumn("period", extract(s, r"([A-Z]+ \d{4}) SUMMARY"))
    .withColumn("unplanned_text", extract(s, r"Unplanned Downtime: ([^|\n]+?)\s*\|"))
    .withColumn("planned_text", extract(s, r"\| Planned: ([^\n]+)"))
    .select(
        "plant_code", "line_name", "period",
        minutes(F.col("unplanned_text")).alias("unplanned_downtime_min"),
        minutes(F.col("planned_text")).alias("planned_downtime_min"),
        extract_num(s, r"Line Availability: ([0-9.]+)%").alias("line_availability_pct"),
        extract_num(s, r"target: ([0-9.]+)%").alias("availability_target_pct"),
        extract_int(s, r"Units Delayed: (\d+)").alias("units_delayed"),
        extract_int(s, r"Scrap: (\d+) units").alias("scrap_units"),
        extract_num(s, r"Scrap: \d+ units \(\$([0-9,]+)\)").alias("scrap_cost_usd"),
        extract_num(s, r"Parts Cost: \$([0-9,]+)").alias("parts_cost_usd"),
        "filename",
        F.current_timestamp().alias("silver_processed_at"),
    )
)
(summary.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{C}.silver.maintenance_summary"))

print("✅ silver.maintenance_events:", events.count(), " silver.maintenance_summary:", summary.count())

In [0]:
#checks, including the reconciliation of the log against its own summary
ev = spark.table(f"{C}.silver.maintenance_events")
sm = spark.table(f"{C}.silver.maintenance_summary").first()

print("entries:", ev.count(), " distinct entry numbers:", ev.select("entry_no").distinct().count())
print("unplanned entries without a root cause:", ev.where("is_unplanned AND root_cause IS NULL").count())
print("entries without a date or a duration:", ev.where(F.col("entry_date").isNull() | (F.col("duration_min") == 0)).count())

ev.select("entry_no", "entry_date", "shift", "equipment_id", "maintenance_type",
          "duration_min", "fault_code", "units_delayed", "scrap_cost_usd",
          "parts_cost_listed_usd").orderBy("entry_no").show(truncate=False)

# Does the log add up to its own summary?
def compare(label, from_entries, from_summary):
    ok = abs(float(from_entries) - float(from_summary)) < 0.01
    print("✅" if ok else "❌", f"{label}: entries={from_entries}  summary={from_summary}")

agg = ev.agg(
    F.sum(F.when(F.col("is_unplanned"), F.col("duration_min")).otherwise(0)).alias("unplanned"),
    F.sum(F.when(~F.col("is_unplanned"), F.col("duration_min")).otherwise(0)).alias("planned"),
    F.sum(F.coalesce("units_delayed", F.lit(0))).alias("units"),
    F.sum(F.coalesce("scrap_cost_usd", F.lit(0.0))).alias("scrap"),
    F.sum(F.coalesce("parts_cost_listed_usd", F.lit(0.0))).alias("parts"),
).first()

compare("Unplanned downtime (min)", agg["unplanned"], sm["unplanned_downtime_min"])
compare("Planned downtime (min)", agg["planned"], sm["planned_downtime_min"])
compare("Units delayed", agg["units"], sm["units_delayed"])
compare("Scrap cost (USD)", agg["scrap"], sm["scrap_cost_usd"])
compare("Parts cost (USD)", agg["parts"], sm["parts_cost_usd"])
print("Line availability:", sm["line_availability_pct"], "% against a target of", sm["availability_target_pct"], "%")
print("entries whose equipment id does not look like an ID:",
      ev.where(~F.col("equipment_id").rlike(r"^[A-Z0-9]+(-[A-Z0-9]+)+$")).count())
ev.select("entry_no", "equipment_id", "equipment_line").orderBy("entry_no").show(truncate=False)